In [1]:
# ===================================================
# CELL 1: SETUP AND IMPORTS
# ===================================================
import os
import sys
import logging
import warnings
import gc
import hashlib
import json
from typing import Tuple, List, Optional, Dict, Any
from pathlib import Path
from dataclasses import dataclass, field
from datetime import datetime
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
from tqdm import tqdm
import cv2
from PIL import Image, ImageOps
import tensorflow as tf
import tensorflow_probability as tfp
from tensorflow.keras import mixed_precision
import matplotlib.pyplot as plt

# Configure environment
logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

print("="*70)
print("ENVIRONMENT VERIFICATION")
print("="*70)
print(f"Python:         {sys.version.split()[0]}")
print(f"TensorFlow:     {tf.__version__}")
print(f"NumPy:          {np.__version__}")
print(f"OpenCV:         {cv2.__version__}")
print("-"*70)

# GPU Configuration
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ GPU memory growth enabled: {len(gpus)} GPU(s)")
        
        gpu_details = tf.config.experimental.get_device_details(gpus[0])
        print(f"   Device: {gpu_details.get('device_name', 'Unknown')}")
        print(f"   Compute: {gpu_details.get('compute_capability', 'Unknown')}")
        
        # Mixed precision
        policy = mixed_precision.Policy('mixed_float16')
        mixed_precision.set_global_policy(policy)
        print("✅ Mixed precision enabled (float16)")
    except Exception as e:
        logger.warning(f"GPU setup: {e}")
else:
    print("⚠️ No GPU detected - using CPU")

# Check OpenCV saliency
if hasattr(cv2, 'saliency'):
    print("✅ OpenCV saliency module available")
else:
    print("⚠️ OpenCV saliency module not found")

print("="*70 + "\n")


ENVIRONMENT VERIFICATION
Python:         3.9.23
TensorFlow:     2.10.1
NumPy:          1.23.5
OpenCV:         4.8.1
----------------------------------------------------------------------
✅ GPU memory growth enabled: 1 GPU(s)
   Device: NVIDIA GeForce RTX 3050 Laptop GPU
   Compute: (8, 6)
INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3050 Laptop GPU, compute capability 8.6


[INFO] Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3050 Laptop GPU, compute capability 8.6


✅ Mixed precision enabled (float16)
✅ OpenCV saliency module available



In [2]:
# ===================================================
# CELL 2: CONFIGURATION
# ===================================================
@dataclass
class Config:
    """Configuration - EDIT THE PATH BELOW"""
    # ✏️ CHANGE THIS TO YOUR DATASET PATH
    raw_root: Path = Path(r"D:\monument_dataset\Indian-monuments")
    out_root: Path = Path(r"D:\monument_dataset\ivp_dataset_processed")
    
    # MEMORY OPTIMIZATION: Maximum image dimension
    max_image_dimension: int = 1024  # REDUCED from 2048 for 16GB RAM
    
    @property
    def saliency_output_path(self) -> Path:
        return self.out_root.parent / "saliency_crops"
    
    @property
    def models_dir(self) -> Path:
        return self.out_root.parent / "models"
    
    @property
    def logs_dir(self) -> Path:
        return self.out_root.parent / "logs"
    
    @property
    def results_dir(self) -> Path:
        return self.out_root.parent / "results"
    
    @property
    def visualizations_dir(self) -> Path:
        return self.out_root.parent / "visualizations"
    
    min_crop_dimension: int = 256
    jpeg_quality: int = 90  # Reduced from 95 for smaller files
    
    def validate_paths(self) -> None:
        if not self.raw_root.exists():
            raise FileNotFoundError(f"Dataset not found: {self.raw_root}")
        if not (self.raw_root / "train").exists() or not (self.raw_root / "test").exists():
            raise FileNotFoundError(f"'train' and 'test' folders required in {self.raw_root}")
    
    def create_directories(self) -> None:
        for dir_path in [self.out_root, self.saliency_output_path, self.models_dir, 
                         self.logs_dir, self.results_dir, self.visualizations_dir]:
            dir_path.mkdir(parents=True, exist_ok=True)
        logger.info("✅ All output directories ready")

# Initialize config
config = Config()
config.validate_paths()
config.create_directories()

print(f"📁 Configuration:")
print(f"   Dataset:     {config.raw_root}")
print(f"   Processed:   {config.out_root}")
print(f"   Max img size: {config.max_image_dimension}px (memory optimized)")
print(f"   Models:      {config.models_dir}")
print(f"   Results:     {config.results_dir}\n")


[INFO] ✅ All output directories ready


📁 Configuration:
   Dataset:     D:\monument_dataset\Indian-monuments
   Processed:   D:\monument_dataset\ivp_dataset_processed
   Max img size: 1024px (memory optimized)
   Models:      D:\monument_dataset\models
   Results:     D:\monument_dataset\results

